<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_30_minimal_runnable_workflow_environment.ipynb">9.30 最小可运行处理工作流</a></li>
        <li>下一节： <a href="9_32_file_level_release_package.ipynb">9.32 文件级示例仓库与发布包</a></li>
    </ul>
</div>


## 9.31 公开归档数据复盘：从科学目标到限制声明

公开归档数据给射电干涉测量教学带来一个很强的实践入口：数据已经存在，质量报告和 pipeline 产品通常也已经存在，复盘过程可以把观测设计、校准、成像、源表、误差预算和可复现工作流放进同一个案例中。归档数据的危险也正在这里。一个数据集能够下载，并不等于它能够回答新的科学问题；一个 pipeline 图像看起来干净，也不等于它保留了目标所需的角尺度、频谱信息、表面亮度灵敏度或系统误差控制。

归档复盘的核心不是重新运行一遍软件，而是重建一条证据链。证据链从科学问题开始，经过元数据筛选、质量报告判读、最小重跑、产品差异审计，最后落到一个清楚的限制声明。结论可能是“可直接使用”，也可能是“重跑后可用”“只能给上限”“仅适合候选筛选”或“不适合”。这些都属于有效的科学结论；真正不可取的是在证据不足时把归档产品当作新的观测结果直接使用。

本节与 9.23、9.27、9.28 和 9.30 形成闭环。9.23 讨论科学目标如何转成观测需求，9.27 讨论 weblog 与 pipeline QA 的证据判读，9.28 和 9.30 讨论项目结构与运行身份。本节把这些内容组织成一个公开归档数据复盘案例，重点放在判断逻辑，而不是某个具体归档网站的按钮流程。


![公开归档数据复盘流程](figures/practical_archive_replay_case_study_flow.png)

图 9.31.1 展示了公开归档数据复盘的主线。每一个向下分支都表示一个可能停止或降级的节点：科学问题本身若无法量化，检索没有意义；覆盖、尺度或频谱不匹配时，后处理不能把缺失信息补回来；质量证据不足时，只能形成候选结论；最小重跑改变科学结果时，需要回到参数与误差预算重新审计。


### 9.31.1 复盘不是重新跑一遍，而是重建证据链

公开归档数据的复盘常被误解为“下载原始数据，按教程重新成像”。这种做法在训练软件操作时有价值，但作为研究流程并不充分。归档数据来自某个原始提案，其频率设置、阵列配置、校准节奏、时间覆盖和 pipeline 参数通常是为原始科学目标服务的。新的科学目标若改变了需要测量的量，也就改变了数据适用性的判据。

一个可审查的复盘至少要回答三类问题。第一类是可观测量问题：目标是测总通量、峰值亮度、谱指数、线宽、旋转量、形态尺度、源数密度，还是只做存在性判定。第二类是数据约束问题：频率、通道宽度、角分辨率、最大可恢复尺度、表面亮度灵敏度、主波束位置、校准证据是否支持这些量。第三类是处理证据问题：归档产品、重跑产品和差异表是否共同支持最终结论。

这里的“证据”不是宽泛的质量印象，而是能够被重新审查的对象：观测元数据、calibrator scan、flag 统计、天气或系统温度记录、bandpass 与 gain 解、权重分布、成像参数、mask、残差图、源表阈值、产品校验和和运行报告。缺少这些对象时，结论就必须降级。归档复盘的成熟标志，不是所有数据都被判为可用，而是每个分支都有可追溯理由。


### 9.31.2 从科学问题到归档检索条件

归档检索不应从望远镜名称或下载格式开始，而应从可观测量开始。设科学目标需要测量某个区域内的连续谱总通量 $S_\nu$、谱指数 $\alpha$、线宽 $\Delta v$ 或角尺度 $\theta_{\rm src}$。这些量会立即给出对数据的硬约束。谱线工作中，观测频率必须覆盖

$$
\nu_{\rm obs}=\frac{\nu_{\rm rest}}{1+z},\qquad
\Delta \nu \simeq \nu_{\rm obs}\frac{\Delta v}{c}.
$$

若目标是线宽或速度场，通道宽度还必须明显小于目标线宽；若目标只是积分通量，通道可以适当合并，但原始频谱覆盖仍要足够。连续谱谱指数工作则要求频率杠杆、带内校准稳定性和主波束模型共同满足要求，否则带内斜率可能混入主波束色散和残余 bandpass 误差。

角尺度约束同样需要在检索阶段完成。合成束近似由最长有效基线决定，最大可恢复尺度由最短有效基线决定：

$$
\theta_{\rm beam}\sim \kappa\frac{\lambda}{B_{\max}},\qquad
\theta_{\rm MRS}\sim \eta\frac{\lambda}{B_{\min}}.
$$

这里 $\kappa$ 与权重和成像细节有关，$\eta$ 常取约 $0.6$ 的量级。若目标角尺度接近或超过 $\theta_{\rm MRS}$，重成像、换权重或调 mask 不能凭空恢复未采样的短间距信息。对于低面亮度目标，还需要把 Jy beam$^{-1}$ 噪声换算为亮温灵敏度：

$$
\sigma_T = \frac{\lambda^2}{2k\Omega_{\rm b}}\sigma_S,\qquad
\Omega_{\rm b}=\frac{\pi\theta_{\rm maj}\theta_{\rm min}}{4\ln 2}.
$$

这个换算解释了一个常见现象：高分辨率图像的 Jy beam$^{-1}$ 噪声看起来很低，但对弥散结构的亮温灵敏度可能并不好。归档检索必须同时记录目标量、所需角尺度、所需频谱分辨率、所需动态范围和允许的系统误差，而不是只记录项目编号。

| 科学量 | 归档检索中必须核对的对象 | 不满足时的后果 |
|---|---|---|
| 连续谱总通量 | 频率、主波束位置、最大可恢复尺度、通量标尺 | 可能系统性漏通量 |
| 峰值亮度与形态 | 阵列配置、合成束、$uv$ 覆盖、残差结构 | 形态可能由波束和旁瓣主导 |
| 谱指数 | 带宽、分频段灵敏度、bandpass 稳定性、主波束模型 | 谱指数可能混入仪器色散 |
| 谱线速度场 | 频率覆盖、通道宽度、速度约定、连续谱扣除空间 | 线宽、速度梯度和 moment 可能偏差 |
| 低面亮度结构 | 短基线、mosaic、taper 后噪声、亮温灵敏度 | 非探测不等于源不存在 |


![归档候选数据集筛选矩阵](figures/practical_archive_replay_fit_matrix.png)

图 9.31.2 把候选数据集按科学需求拆成若干判断项。绿色区域表示该列的主要依据，黄色区域表示必须进一步检查的证据。矩阵的用途不是把数据集机械打分，而是分清哪些限制可以通过重成像、重校准或限定区域缓解，哪些限制属于观测本身没有提供的信息。


### 9.31.3 候选数据集筛选：哪些限制能补救，哪些不能

归档数据的限制大致可分为三类。第一类是观测信息缺失，例如目标谱线没有落入频窗、目标尺度超过最大可恢复尺度、目标区域位于主波束外很远的位置、校准源扫描根本不足。这类限制通常不能由后处理补回。第二类是处理选择不匹配，例如 pipeline 图像使用了过高分辨率的 Briggs 权重、clean mask 过保守、自动阈值对低面亮度结构不友好、源表阈值没有按科学目标设置。这类限制常可通过重成像或重新生成产品缓解。第三类是证据不充分，例如质量报告缺少关键图、flag 版本不明、pipeline 参数不可追溯、产品没有校验和。这类限制不一定说明数据不可用，但会降低结论等级。

因此，筛选候选数据集时应先判定“硬不匹配”，再考虑“软不匹配”。硬不匹配直接决定科学目标是否需要改写；软不匹配决定是否值得投入重跑。以低面亮度连续谱为例，阵列缺少足够短基线是硬限制，pipeline 图像没有使用 taper 是软限制。前者会使非探测不能解释为物理上没有弥散辐射，后者则可以通过重成像形成新的产品。

一个简洁的筛选记录可以保留为表格，而不是散落在文字描述中。表格的关键不是列数多，而是每一行都能指向证据来源。

| 判断项 | 证据来源 | 可补救性 | 复盘动作 |
|---|---|---|---|
| 频率覆盖 | spectral window 表、目标红移、线频 | 通常不可补救 | 不覆盖则停止或改写问题 |
| 最大可恢复尺度 | 阵列配置、最短有效基线、flag 后 $uv$ 覆盖 | 通常不可补救 | 只能给小尺度结论或上限 |
| 图像权重与 taper | pipeline 参数、原图 beam、权重表 | 常可补救 | 用目标权重重成像 |
| 自动 mask | clean 日志、残差图、模型图 | 常可补救 | 手动或半自动重设 mask |
| 通量标尺 | flux calibrator、标准模型、gain 解 | 部分可补救 | 核对标尺并纳入系统误差 |
| 主波束位置 | pointing、mosaic 表、primary beam 图 | 部分可补救 | 限定区域或增加主波束误差 |
| flag 历史 | flag 版本、人工 flag 记录、QA 报告 | 取决于原始数据 | 复现 flag 或保留限制声明 |

筛选阶段的产物应是“候选数据集说明”，而不是最终处理脚本。说明中需要明确哪些数据进入下一步，哪些数据被排除，排除原因属于硬限制、软限制还是证据不足。这样后续重跑不会变成盲目试错。


### 9.31.4 质量报告与 weblog 证据板

现代大型阵列的归档产品常附带 pipeline weblog、质量报告或处理摘要。这些文件不应只被读成“通过”或“警告”。同一条警告对不同科学目标权重不同。相位校准解的一次短时跳变，对高动态范围成像可能很严重；对只需粗略上限的非探测分析可能可以进入系统误差。某个频窗边缘 bandpass 残差明显，若科学目标正好依赖该频窗边缘的谱线，就必须降级；若目标只使用中心连续谱窗口，影响可能很小。

证据板的作用是把质量报告的文字和图像重新映射到科学目标。它通常包括元数据、标记摘要、校准解、天气或系统温度、图像产品、处理参数、人工记录和科学区域。每个证据块都要回答同一个问题：它怎样影响目标量的可信度。


![质量报告与 weblog 证据板](figures/practical_archive_replay_weblog_evidence_board.png)

图 9.31.3 中央的证据板表示复盘过程的核心对象。元数据、标记、校准解、天气与系统状态、图像产品、处理参数、人工记录和科学区域都指向同一个科学目标。证据板不是把文件全部复制到报告里，而是把每条质量信息翻译成对目标量的影响。


在实践中，weblog 判读至少应覆盖五个层次。元数据层检查日期、阵列配置、频窗、扫描、目标位置和校准源。标记层检查 flag 比例是否集中在特定天线、基线、时间段或频窗；随机少量 flag 与系统性短基线丢失的科学含义完全不同。校准层检查 bandpass、gain、delay、fluxscale 的解是否平滑、是否有不合理跳变、是否依赖过短解间隔。成像层检查 beam、噪声、残差、动态范围、主波束校正区域和自动 mask 行为。处理历史层检查 pipeline 版本、人工改动、失败任务、重跑尝试和产品身份。

证据板可以直接写入复盘报告。例如，某个目标需要 30 arcsec 尺度的低面亮度连续谱测量，weblog 显示短基线大量 flag 且 pipeline 图像使用 3 arcsec beam。这里有两个不同层次的问题：短基线 flag 影响可恢复尺度，属于观测证据限制；3 arcsec beam 影响亮温灵敏度，属于可重成像的处理限制。把二者混在一起，只写“图像噪声较高”，会丢失真正的判断结构。

质量报告中的“通过”也不是最终答案。自动 QA 往往检查通用标准，例如校准解是否存在、图像噪声是否接近期望、少数指标是否超阈值。新的科学目标可能要求比通用 QA 更严格的条件，也可能只依赖通用 QA 没有覆盖的量。归档复盘必须把自动 QA 翻译成目标 QA。


### 9.31.5 最小重跑：保留校准还是重新成像

归档复盘进入重跑阶段时，原则是最小充分改变。若科学问题只要求改变成像权重、taper、mask 或阈值，而校准证据充分，则优先保留归档校准，重新成像并记录参数。若质量报告显示 bandpass、gain 或 fluxscale 与科学目标直接冲突，才进入重校准。若原始数据、flag 历史或校准源证据不足以支撑重校准，则重跑不能制造可信度，只能降低结论等级。

这种层次化选择有两个原因。第一，校准会改变数据的复增益标尺，影响所有后续产品；不必要的重校准会引入新的自由度，使差异难以解释。第二，成像参数更直接地连接科学目标，例如从高分辨率点源图像改为低面亮度结构图像时，通常首先改变权重、uv taper、多尺度 clean 和 mask，而不是重求所有校准解。

最小重跑包应保留原产品、重跑产品和差异表。一次归档连续谱重跑可以组织为如下结构：

```text
archive_replay_project/
  manifests/
    archive_data.yaml
    products_original.yaml
    products_rerun.yaml
  qa/
    weblog_evidence_board.md
    calibration_notes.md
    imaging_validation.md
  configs/
    imaging_low_surface_brightness.yaml
    source_finding.yaml
  runs/
    run_2026_05_04/
      run_report.md
      command_log.txt
      product_manifest.yaml
  products/
    original/
    rerun_tapered/
    comparisons/
```

这里的目录结构继承 9.28 和 9.30 的思想：大文件不一定进入版本库，但身份、参数和验证结论必须进入项目文本。每一次重跑都需要回答输入是否相同、环境是否可识别、参数是否改变、产品是否通过验证。若某个中间产品来自手动交互操作，也必须在运行报告中说明，因为交互判断往往正是低面亮度和复杂残差处理中最影响结论的部分。


### 9.31.6 原产品与重跑产品的差异审计

重跑完成后，不能只展示一张“更好看”的图。归档复盘需要逐项比较原产品和重跑产品，并解释差异来自输入、参数、权重、mask、校准还是选择函数。比较对象至少包括合成束、图像噪声、残差结构、通量恢复、峰值位置、源表数量、局部背景和科学区域内的积分量。

图像差异可以从几个简单量开始。若两幅图已经卷积到共同 beam，差异图

$$
D(l,m)=I_{\rm rerun}(l,m)-I_{\rm original}(l,m)
$$

可以显示重跑究竟改变了科学区域、旁瓣残差还是背景零点。若目标是积分通量，需比较

$$
S_A=\sum_{(l,m)\in A} I(l,m)\,\Delta\Omega / \Omega_{\rm b},
$$

并把噪声相关性、主波束校正、通量标尺和区域选择纳入误差预算。若目标是源表，差异不应只看源数增减，还要看新增源是否集中在主波束边缘、残差旁瓣附近或噪声不均匀区域。若目标是谱线 cube，moment 图差异必须回到 mask、连续谱扣除和通道噪声，而不是直接解释为气体分布差异。


![原产品与重跑产品差异审计](figures/practical_archive_replay_product_comparison.png)

图 9.31.4 强调原产品、重跑产品和差异表三者必须同时保留。差异表应记录 beam、噪声、通量、残差、源数和参数身份。差异不会自动说明哪一个产品正确；它需要回到输入证据、处理参数、权重选择和科学目标共同解释。


差异审计中最容易忽视的是“比较前是否可比”。两个图像若 beam 不同，直接逐像素相减会把分辨率差异误当成物理结构。两个源表若检测阈值、背景估计或岛分裂规则不同，源数差异可能来自选择函数，而不是天空变化。两个谱线 moment 图若 mask 不同，外层低信噪比结构可能只是 mask 策略的产物。

因此，复盘报告应明确说明比较前的统一步骤：是否卷积到共同 beam，是否裁剪到同一主波束范围，是否使用相同像素网格，是否采用相同区域，是否把单位统一到 Jy beam$^{-1}$、Jy pixel$^{-1}$ 或 K，是否将源表阈值和可靠性估计固定。只有这些条件满足后，产品差异才具有科学含义。


### 9.31.7 限制声明：适合、部分适合、不适合都是结论

归档复盘的最后一步是限制声明。限制声明不是报告末尾的礼貌性文字，而是把科学结论限定在证据能够支持的范围内。一个成熟的限制声明会同时说明数据能够回答什么，不能回答什么，哪些结论依赖重跑参数，哪些结论受观测本身限制。

限制声明通常落在五个等级。最高等级是可直接使用，表示归档产品和证据已经满足科学目标。第二级是重跑后使用，表示观测信息充足，但原处理参数不适合，重跑产品通过验证。第三级是部分可用，表示只在限定区域、频段、角尺度或信噪比范围内可靠。第四级是只能给限制，表示数据不足以给出检测或精确测量，但可给出上限、非检测约束或候选判断。最低等级是不可用，表示关键频率、尺度或校准信息缺失，后处理无法补救。


![归档复盘限制声明阶梯](figures/practical_archive_replay_limit_statement_ladder.png)

图 9.31.5 把限制声明组织成阶梯。成熟的复盘不强迫所有数据给出肯定结果；清楚的限制、上限、非检测和候选结论同样具有科学价值。报告、发布包和后续工作都应由这个等级判断导出。


一个有用的限制声明可以写成这样的逻辑结构：目标量是什么；使用了哪些归档数据和产品；哪些 QA 证据支持使用；进行了哪些最小重跑；原产品与重跑产品在哪些量上一致或不同；最终结论适用于哪些角尺度、频率范围、主波束区域和信噪比范围；哪些物理解释被排除，哪些仍不能排除。

例如，某个连续谱归档复盘若发现 taper 后科学区域内没有显著弥散辐射，但最大可恢复尺度低于目标预期尺度，则结论不能写成“未检测到弥散辐射，因此源不存在”。更合适的结论是：在某一 beam、频率、主波束范围和表面亮度阈值下，未检测到小于可恢复尺度的弥散结构；更大尺度或更低面亮度成分仍不受该数据限制。这样的表述看似保守，却是可审查研究的基础。


### 9.31.8 教学案例：低面亮度连续谱归档复盘

考虑一个典型教学案例：目标是评估一个近邻星系团外围是否存在低面亮度连续谱弥散辐射。已有公开归档数据包括一个 L 波段连通阵列观测，pipeline 图像的角分辨率约为数角秒，局部 rms 接近归档报告预期。单看 pipeline 图像，核心致密源清楚，外围区域没有明显弥散结构。这个结果不能直接解释为非探测，因为科学目标真正依赖的是几十角秒到数角分尺度的亮温灵敏度和短基线覆盖。

第一步是把科学问题量化。目标不是“图像里有没有雾状结构”，而是在给定区域内测量或限制弥散辐射的积分通量、平均表面亮度和空间尺度。于是检索条件必须包括频率覆盖、短基线、目标距 pointing center 的角距离、预期弥散尺度、可接受 beam、taper 后噪声和主波束误差。若预期结构为 $2'$ 到 $4'$，而 flag 后最大可恢复尺度只有 $1'$ 左右，则数据只能约束小尺度子结构，不能约束完整弥散源。

第二步是建立 weblog 证据板。若质量报告显示整体校准通过，但短基线在某些扫描被大量 flag，且 pipeline 图像采用接近均匀权重以优化致密源分辨率，那么证据板应把这两点分开记录：短基线损失是观测与 flag 后的硬限制；高分辨率权重是可重成像的软限制。同时检查 flux calibrator、phase calibrator、bandpass 解和残差图，确认是否存在会影响低面亮度结构的系统误差，例如目标场内强源旁瓣、方向相关残差或主波束边缘噪声抬升。

第三步是做最小重跑。保留原校准，先用自然权重或 robust 加 uv taper 重新成像，并使用多尺度 clean 和与科学区域相容的 mask。重跑后把图像卷积到共同 beam，裁剪到相同主波束范围，比较原图、taper 图和差异图。若 taper 后局部 rms 上升但亮温灵敏度改善，报告应同时列出 Jy beam$^{-1}$ 噪声和 K 单位亮温灵敏度。若核心致密源残差限制了外围区域，必须把动态范围限制而非热噪声写入误差预算。

第四步是形成限制声明。若重跑产品在目标区域仍无显著弥散信号，并且可恢复尺度覆盖了目标尺度的一部分，可以给出该尺度范围内的表面亮度上限。若最大可恢复尺度明显不足，则只能说明没有检测到小于某一尺度的成分，不能排除更大尺度辐射。若主波束校正后区域噪声不均匀，源表或通量上限还需限定在主波束响应高于某个阈值的区域内。这个案例的教学价值在于：同一个归档数据集可以同时给出有效上限、无效大尺度结论和需要新观测的理由。

最终报告可以用一段短结论闭合证据链：归档观测的频率和校准证据足以支持 L 波段连续谱复盘；原 pipeline 图像不适合低面亮度目标，因为权重和 beam 偏向致密源；taper 重跑改善了目标尺度的亮温灵敏度；短基线 flag 和最大可恢复尺度限制了对更大弥散结构的解释；因此，该数据只支持某一角尺度和表面亮度阈值内的非检测或上限，不能替代针对更大尺度设计的新观测。


### 9.31.9 本节结论

公开归档数据复盘把实践训练从“会运行软件”推进到“会建立证据链”。科学问题决定检索条件，元数据和质量报告决定数据是否进入下一步，最小重跑负责让处理参数匹配科学目标，产品差异审计负责解释重跑改变了什么，限制声明负责把结论限定在证据能够支持的范围内。

归档数据的可用性不是二元判断。频率、角尺度、表面亮度灵敏度、校准证据、主波束区域和处理历史共同决定最终等级。能够清楚说明直接适合、重跑后适合、部分适合、只能给限制或完全不适合，是公开数据训练中最接近真实研究的一环。
